# 00 — Setup and sanity check

Gets the detector stack (HierarchicalDet: DiffusionDet + Detectron2, Swin-Large
backbone) running on this Kaggle notebook, and confirms the whole pipeline
works end to end on a *tiny* amount of data before you spend GPU hours on the
real training run in `01_train_baseline_detector.ipynb`.

**Status of what's below**: every step was verified locally (macOS arm64,
CPU/MPS, no GPU) in the project's development session — see `SETUP.md` in the
repo for the exact recipe this notebook follows. The install steps are
adapted here for Kaggle's environment (skip reinstalling torch — use the
image's preinstalled build). This notebook itself has not been run on actual
Kaggle hardware yet — if a cell fails, the error message plus `SETUP.md`
should get you unstuck fast; please update this notebook with what you find
so the next person doesn't hit the same thing.

**Before running**: in the Kaggle notebook settings, turn on GPU (T4 x2 or
P100) and internet access.

## 1. Clone the repo and install dependencies (idempotent)

In [ ]:
# Idempotent bootstrap -- safe to re-run (session restart, or you ran the cell twice).
# The old version was `!git clone` + `%cd`: it errored on the second run, and then
# left the notebook in the wrong directory with every relative path quietly broken.
import os, subprocess, sys

REPO_URL = "https://github.com/christopherh-88/Carries-Confidence.git"
if not os.path.exists("src/data/degradation.py"):          # not already at the repo root
    if not os.path.isdir("Carries-Confidence"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("Carries-Confidence")
sys.path.insert(0, ".")

from src.utils.kaggle_env import install_deps, summarize_environment

# Installs ONLY what's missing, with numpy/torch pinned to the image's versions.
# Do NOT `pip install -r requirements-core.txt` here: it can upgrade numpy, and
# Kaggle's torch -- plus the detectron2 you are about to build against it -- is
# compiled for the numpy already in the image. The upgrade "succeeds", then torch
# dies at import with "compiled using NumPy 1.x cannot be run in NumPy 2.x".
print("installed:", install_deps() or "nothing needed -- image already has it")
for k, v in summarize_environment().items():
    print(f"  {k}: {v}")

## 2. Confirm the GPU

Do NOT `pip install torch`/`torchvision` — Kaggle's image ships a specific
torch+CUDA build matched to the driver, and replacing it is the #1 way to
break the compiled ops later (see SETUP.md). The bootstrap above pins torch
and numpy so nothing moves them by accident.

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("no GPU in this session -- fine for setup, but notebook 01 needs one")

## 3. Clone the HierarchicalDet baseline

In [ ]:
!bash scripts/clone_baseline.sh

## 4. Install detectron2

`--no-build-isolation` is required: detectron2's setup.py does `import torch`
at build time and pip's isolated build environment cannot see it otherwise.
This is by far the slowest step in the notebook.

In [ ]:
# Build detectron2 against the image's torch. Never reinstall torch on Kaggle.
# Deliberately NOT -q: this compiles for ~10 minutes, and a silent cell that long
# is indistinguishable from a hang.
import torch
print(f"building detectron2 against torch {torch.__version__} -- expect ~10 min")

!pip install -q ninja
!pip install --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"

## 5. Confirm the compiled ops load

If this fails on `_C`, STOP — nothing downstream works until it resolves. It
almost always means torch changed after detectron2 was built.

In [ ]:
import detectron2
from detectron2 import _C
print("detectron2", detectron2.__version__, "+ compiled ops (_C): OK")

## 6. Import HierarchicalDet against the REAL detectron2

HierarchicalDet vendors its own `detectron2/` (no compiled ops) and
`pycocotools/` (no compiled `_mask`). If its directory reaches `sys.path`
before the real packages are imported, `import detectron2` silently resolves
to the broken vendored copy. `import_hierarchicaldet()` enforces the order.

In [ ]:
from src.utils.kaggle_env import import_hierarchicaldet

# Imports the real detectron2/pycocotools BEFORE putting HierarchicalDet on
# sys.path, so its vendored (uncompiled) copies cannot shadow them. Raises a
# clear error if the compiled ops are missing, instead of failing obscurely later.
add_diffusiondet_config = import_hierarchicaldet()
print("hierarchialdet imports OK, using the real detectron2")

## 7. Download and convert the Swin-Large backbone weights

**Read this before running**: the config's own weights filename
(`swin_base_patch4_window7_224_22k.pkl`) is misleading — it implies
Swin-Base, but the config's `MODEL.SWIN.SIZE` is `L-22k` (Swin-**Large**).
DiffusionDet's official release only ships Swin-Base weights under that
filename; using them here would silently fail to load ~90% of the backbone
(detectron2's checkpointer warns per-tensor but does not raise an error).

Confirmed fix: download the actual Swin-Large-22k classification checkpoint
from Microsoft's official release and convert it to detectron2's format.

In [ ]:
import os
os.makedirs("models_weights", exist_ok=True)

# Guarded: the raw checkpoint is ~900MB, and this cell used to re-download it
# every single run (including after a session restart, when the converted .pkl
# is already sitting there from an attached dataset).
PKL = "models_weights/swin_large_patch4_window7_224_22k.pkl"
if os.path.exists(PKL):
    print("already converted, skipping download:", PKL)
else:
    !curl -sL --fail "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_large_patch4_window7_224_22k.pth" \
      -o models_weights/swin_large_patch4_window7_224_22k_raw.pth

import torch, pickle

if not os.path.exists(PKL):
    ckpt = torch.load("models_weights/swin_large_patch4_window7_224_22k_raw.pth", map_location="cpu", weights_only=False)
    assert ckpt["model"]["patch_embed.proj.weight"].shape[0] == 192, "expected Swin-Large's 192-dim embedding, got something else -- did the download change?"
    converted = {"model": ckpt["model"], "__author__": "third_party", "matching_heuristics": True}
    with open("models_weights/swin_large_patch4_window7_224_22k.pkl", "wb") as f:
        pickle.dump(converted, f)
    os.remove("models_weights/swin_large_patch4_window7_224_22k_raw.pth")  # save disk space, no longer needed
    print("converted OK")

assert os.path.exists(PKL), "weights conversion did not produce " + PKL
print("backbone weights ready:", PKL)

## 8. Point at the DENTEX dataset

Attach it via "Add Data" in the sidebar — see `docs/phase2_data_notes.md` for
how to upload it, and why the Hugging Face download does not translate well
to a Kaggle session. The lookup below searches `/kaggle/input` for the
annotation file rather than assuming a dataset name.

If you would rather download it in-session: set `HF_TOKEN` as a Kaggle secret
(Settings → Add-ons → Secrets), then run `scripts/download_dentex.py` and
unzip `training_data/quadrant-enumeration-disease/*` before this cell.

In [ ]:
from src.utils.kaggle_env import find_dentex_root

# Finds DENTEX wherever it is mounted rather than assuming a dataset slug -- the
# path depends on what you named the Kaggle Dataset. If it is not attached, the
# error tells you how to attach it.
DATA_ROOT = str(find_dentex_root())
print("DATA_ROOT:", DATA_ROOT)

## 9. Build the real model and confirm the checkpoint loads cleanly

This is the check that catches the Swin-Base/Large mismatch bug from step 7
if it recurs. **Zero** "will not be loaded" lines should mention any
`backbone.bottom_up.*` key. Lines about `head.*`, FPN lateral/output convs,
or diffusion-schedule buffers (`alphas_cumprod` etc.) are expected and fine
— those aren't part of an ImageNet classification checkpoint.

In [ ]:
from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer

cfg = get_cfg()
add_diffusiondet_config(cfg)
cfg.merge_from_file("external/HierarchicalDet/configs/diffdet.custom.swinbase.nonpretrain.yaml")
cfg.MODEL.WEIGHTS = "models_weights/swin_large_patch4_window7_224_22k.pkl"
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(cfg)
DetectionCheckpointer(model).load(cfg.MODEL.WEIGHTS)
n_params = sum(p.numel() for p in model.parameters())
print(f"model built OK: {type(model).__name__}, {n_params:,} params, device={cfg.MODEL.DEVICE}")
print()
print("^^ scroll up: confirm no 'will not be loaded' lines mention backbone.bottom_up.*")

## 10. Forward pass and one real training step (tiny sanity check)

Uses 2 real DENTEX images, not synthetic data — the actual target for
"confirm a baseline forward pass" before committing to the full run. Fast
regardless of GPU/CPU since it's only 2 images once.

In [ ]:
import copy
import cv2
import numpy as np
from detectron2.structures import Instances, Boxes

sys.path.insert(0, ".")  # repo root, for src.*
from src.data.dentex import load_coco, patient_level_split, register_dentex_detectron2
from detectron2.data import DatasetCatalog

coco = load_coco(f"{DATA_ROOT}/train_quadrant_enumeration_disease.json")
split = patient_level_split(coco, seed=0)
register_dentex_detectron2(coco, f"{DATA_ROOT}/xrays", split)
train_dicts = DatasetCatalog.get("custom_train_class")
print("registered custom_train_class:", len(train_dicts), "images")

def simple_mapper(d, target_size=800, device="cpu"):
    d = copy.deepcopy(d)
    img = cv2.imread(d["file_name"], cv2.IMREAD_COLOR)
    h0, w0 = img.shape[:2]
    img = cv2.resize(img, (target_size, target_size))
    scale_x, scale_y = target_size / w0, target_size / h0

    inst = Instances((target_size, target_size))
    boxes, c1, c2, c3 = [], [], [], []
    for ann in d["annotations"]:
        x, y, w, h = ann["bbox"]
        boxes.append([x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y])
        c1.append(ann["category_id_1"]); c2.append(ann["category_id_2"]); c3.append(ann["category_id_3"])
    inst.gt_boxes = Boxes(torch.tensor(boxes, dtype=torch.float32)) if boxes else Boxes(torch.zeros(0, 4))
    inst.gt_classes_1 = torch.tensor(c1, dtype=torch.int64)
    inst.gt_classes_2 = torch.tensor(c2, dtype=torch.int64)
    inst.gt_classes_3 = torch.tensor(c3, dtype=torch.int64)

    return {
        "image": torch.as_tensor(img.transpose(2, 0, 1).astype(np.float32)).to(device),
        "height": target_size, "width": target_size,
        "instances": inst.to(device),
    }

device = cfg.MODEL.DEVICE
batch = [simple_mapper(train_dicts[0], device=device), simple_mapper(train_dicts[1], device=device)]
print("batch built, boxes per image:", [len(b["instances"]) for b in batch])

model.eval()
with torch.no_grad():
    out = model(batch)
print("inference OK, instances predicted:", [len(o["instances"]) for o in out])

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)
loss_dict = model(batch)
loss = sum(loss_dict.values())
loss.backward()
optimizer.step()
print("training step OK, losses:", {k: round(v.item(), 3) for k, v in list(loss_dict.items())[:5]})
print()
print("=== SETUP VERIFIED. Proceed to 01_train_baseline_detector.ipynb ===")